# Sequence Classification with BART (Sentiment Analysis)

**Objective:** Fine-tune the `facebook/bart-base` model on the IMDB dataset for binary sentiment classification (Positive/Negative). 

**Why BART?**
BART (Bidirectional and Auto-Regressive Transformers) is an **Encoder-Decoder** architecture. 
- The **Encoder** bidsirectionally reads the entire corrupted document (like BERT) to understand deep context.
- The **Decoder** auto-regressively predicts the output (like GPT).
While historically loved for seq2seq tasks like summarization, BART is incredibly powerful for sequence classification. To classify, BART feeds the same text to both the encoder and decoder, and attaches a dense classification head directly to the final `EOS` (End-of-Sentence) hidden state of the decoder. This yields remarkably robust categorizations for varying sequence lengths.

### Notebook Phases
| Phase | Description |
|---|---|
| **Phase 1** | Environment & Dependencies (transformers, evaluate, datasets) |
| **Phase 2** | Data Engineering (Loading IMDB, tokenization, formatting) |
| **Phase 3** | Model Architecture (BART Config, Class Mappings) |
| **Phase 4** | Training & Evaluation (Custom Trainer with Class Weights, Metrics Tracking) |
| **Phase 5** | Inference Pipeline (Human-readable text prediction) |

---
## Phase 1: Environment & Dependencies
To begin, we ensure that our environment possesses the right Hugging Face libraries for datasets and metric evaluation.

### 📖 Guide: Installing the Hugging Face Stack

- `transformers`: Provides the `BartTokenizer` and `BartForSequenceClassification` models.
- `datasets`: The fastest way to pull the standardized IMDB dataset.
- `evaluate` & `scikit-learn`: Crucial for tracking dynamic metrics like F1 and Accuracy during training.
- `accelerate`: The underlying PyTorch orchestrator to seamlessly manage GPU training.

In [ ]:
# --- Install Required Libraries ---
!pip install -q transformers datasets evaluate accelerate torch scikit-learn

In [ ]:
# --- Core Imports ---
import torch                                          # PyTorch deep learning framework
from transformers import (                            # Hugging Face transformers ecosystem
    BartTokenizer, 
    BartForSequenceClassification, 
    Trainer, 
    TrainingArguments
)
from datasets import load_dataset                     # For handling HF datasets
import evaluate                                       # HF library for standardized metric calculation
import numpy as np                                    # For scientific array operations

---
## Phase 2: Data Engineering
We'll download a subset of the IMDB dataset, tokenize it precisely for the BART architecture, and inject it into PyTorch Tensors.

### 📖 Guide: Loading Datasets and Robust Tokenization

BART handles inputs specifically with <s> and </s> tokens. 
We instruct our tokenizer to dynamically `truncate` sentences to 512 tokens (the max context limit for most standard models), and apply `padding` to ensure uniform matrix dimensions, which is an absolute requirement for feeding text batches natively to the GPU.

In [ ]:
# --- Load Dataset ---
# We slice the dataset to 2,000 training nodes and 1,000 testing nodes for speed
dataset = load_dataset("imdb", split={"train": "train[:2000]", "test": "test[:1000]"})

# --- Initialize Tokenizer ---
# Instantiate the standard BART tokenizer
tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")

def tokenize_function(batch):
    # Max_length=512 ensures we do not exceed memory limits. 
    # Padding applies zeros up to the batch's longest sequence.
    return tokenizer(batch["text"], padding=True, truncation=True, max_length=512)

# --- Parallel Mapping ---
# Apply tokenization row-by-row efficiently leveraging batching
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Let's verify our processed columns:
print("Columns:", tokenized_dataset["train"].column_names)

In [ ]:
# --- PyTorch Formatting ---
# The Trainer expects labels to be named 'labels' and tensors format to be native 'torch'.
if "label" in tokenized_dataset["train"].column_names:
    tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
    
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

---
## Phase 3: Model Architecture
We need to fetch the pretrained weights for BART and instruct the architecture that only 2 final output neuro-paths exist (Positive/Negative).

### 📖 Guide: Initializing `BartForSequenceClassification`

During fine-tuning, `BartForSequenceClassification` drops its native language-modeling "head" and securely bolts on a dense linear classification head with exactly `num_labels=2` outputs. 

We also explicitly feed `id2label` to automatically map internal integers `0` and `1` directly back to English sentiment terms during inference later on.

In [ ]:
# --- Load Model Layout ---
# Explicitly mapping IDs to their string variants heavily assists the API outputs later
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = BartForSequenceClassification.from_pretrained(
    "facebook/bart-base", 
    num_labels=2,          # Output dimensions
    id2label=id2label,     # 1 -> POSITIVE
    label2id=label2id      # POSITIVE -> 1
)

# Identify if GPU is available to bypass slow CPU constraints
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)           # Push whole model to VRAM completely

print(f"Model allocated to -> {device}")

---
## Phase 4: Training & Evaluation
To properly scale training, we inject Accuracy and F1 computational tools, establish evaluation strategies per epoch, and define handling for class weights to improve robustness.

### 📖 Guide: Dynamic Classification Metrics & Custom Class Weight Trainer

Training naturally runs on Cross-Entropy loss. However, humans judge models on **Accuracy** and **F1 Scores**. We define `compute_metrics` so the Trainer computes these metrics on the `eval_dataset` at the end of every epoch.

***Optimization Highlight:*** If a dataset is unbalanced (e.g., 90% positive, 10% negative), a standard model biases heavily toward predicting positive. We override HuggingFace's Trainer class via `CustomClassWeightTrainer` to inject PyTorch `.CrossEntropyLoss(weight=...)`, financially penalizing the loss heavily whenever it gets the minority class wrong. *(Note: IMDB is perfectly balanced so weights are [1.0, 1.0], but this structure provides an educational production template).* 

In [ ]:
# --- Defines Objective Accuracy Checkers ---
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    # Extracts pure logits array and true label array from trainer evaluation prediction loop
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)  # Collapse the highest probability logit to an absolute index (0 or 1)
    
    # Mathematically compute accuracy & F1 score based on ground truth
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1}


# --- Setup Class Weights Optimization ---
import torch.nn as nn

class CustomClassWeightTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Pop the explicit label tags so the model processes inputs unguided
        labels = inputs.pop("labels")
        
        # Perform the actual forward prediction pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Supply explicit weights for each class tensor mapping. e.g [1.0, 5.0] would prioritize predicting Class '1'.
        # Sending to the active cuda device
        class_weights = torch.tensor([1.0, 1.0]).to(model.device) 
        
        # Use CrossEntropyLoss heavily punishing model based on the above tensor weights
        loss_function = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_function(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

### 📖 Guide: Execute `TrainingArguments`
We set `eval_strategy="epoch"` and `logging_strategy="steps"` so we can dynamically compare how effectively the Loss goes down and the metrics go up per phase.

In [ ]:
# --- Build Trainer Engine ---
training_args = TrainingArguments(
    output_dir="./bart_sentiment_results",    # Local drive persistence location
    eval_strategy="epoch",                    # Trigger model testing after every full data pass
    learning_rate=2e-5,                       # Micro steps across the gradient plane to prevent sudden instability
    per_device_train_batch_size=8,            # Maximum batch chunks handled by GPU concurrently
    per_device_eval_batch_size=16,            # Evaluation takes less memory so we increase processing batches
    num_train_epochs=3,                       # Three total passes on datasets
    weight_decay=0.01,                        # Regularization parameter dynamically clamping heavy outlier weights
    logging_strategy="steps",                 # Report active metrics per steps
    logging_steps=50,                         # Output terminal loss telemetry every 50 batches
    report_to="none"                          # Deactivate W&B integrations for offline stability and simplicity
)

trainer = CustomClassWeightTrainer(
    model=model,                              # Target BART architecture
    args=training_args,                       # Active hyperparameters
    train_dataset=tokenized_dataset["train"], # Input Train Set
    eval_dataset=tokenized_dataset["test"],   # Input Validation Set
    compute_metrics=compute_metrics           # Attach the custom F1/Acc handler logic
)

# Start loop computing tensors and updating architecture
trainer.train()

---
## Phase 5: Inference Pipeline
We lock in model weights (`torch.no_grad()`) and supply arbitrary English text to evaluate the new system.

### 📖 Guide: Text to Sentiment Extraction
We pass raw text back into our `tokenizer` to convert to ID Tensors. `torch.softmax` collapses the RAW vector logit predictions evenly across a percentage totaling 1.0 (e.g 95% Positive, 5% Negative). We output an understandable JSON inference prediction.

In [ ]:
# --- Setup Generation Pipeline ---
def predict_sentiment(text):
    # 1. Transform sequence string onto identical layout used during training
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    
    # 2. Deactivate AutoGrad engines. Significantly accelerates processing given we no longer need to update weights
    with torch.no_grad():
        outputs = model(**inputs)
        
    # 3. Process Predictions
    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=1).detach().cpu().numpy()[0]   # Get clean percentages [0-1]
    predicted_class_id = torch.argmax(logits, dim=-1).item()                 # Extract highest probability index
    predicted_label = model.config.id2label[predicted_class_id]              # Reverse MAP: 1 -> POSITIVE
    
    # 4. Construct human-readable diagnostic breakdown
    return {
        "text": text,
        "sentiment": predicted_label,
        "confidence_score": f"{np.max(probabilities) * 100:.2f}%",
        "class_probabilities": {
            "NEGATIVE": f"{probabilities[0] * 100:.2f}%",
            "POSITIVE": f"{probabilities[1] * 100:.2f}%"
        }
    }

# --- Run Inference Validation ---
sample_1 = "This movie was an absolute disaster, completely boring and predictable."
sample_2 = "This film was fantastic! I loved every single moment of it."

print("Sample 1 Prediction:\n", predict_sentiment(sample_1), "\n")
print("Sample 2 Prediction:\n", predict_sentiment(sample_2))

---
# Appendix: Adapting This Pipeline for Other Models

This pipeline works for **any Encoder Model**. Three things typically change:

---

### A — LoRA Target Modules
Unlike LLMs, Encoder models have slightly different naming conventions for their Attention layers. If you want to use PEFT/LoRA, aim for these modules:

| Model Family | `target_modules` |
|-------------|------------------|
| **BERT / RoBERTa** | `query, value` (or `query, key, value`) |
| **DeBERTa** | `query_proj, key_proj, value_proj` |
| **DistilBERT** | `q_lin, k_lin, v_lin` |
| **BART** | `q_proj, v_proj` |

---

### B — Tokenizer Context Handling
Unlike Causal LLMs which require manual "Prompt Templates" (`[INST]`, `<|im_start|>`), Encoder models handle structure automatically via special tokens when you call the tokenizer. You just need to be aware of their context window limits.

| Model | Special Tokens | Max Sequence Length |
|-------|----------|-------------------|
| **BERT / DistilBERT** | `[CLS] {text} [SEP]` | 512 |
| **RoBERTa / BART** | `<s> {text} </s>` | 512 |
| **Longformer** | `<s> {text} </s>` | 4096 |

---

### C — Model Loader Class
When tackling Encoder tasks, the AutoModel class you import changes depending on the specific NLP objective.

| Task Type | Class | Example |
|-----------|-------|---------|
| **Sequence Classification** | `AutoModelForSequenceClassification` | Sentiment Analysis, Spam Detection |
| **Token Classification** | `AutoModelForTokenClassification` | Named Entity Recognition (NER) |
| **Question Answering** | `AutoModelForQuestionAnswering` | Extractive Q&A |
